<a href="https://colab.research.google.com/github/saranoor/chatGPT/blob/master/policy_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Load the dataset
We will combine the review summary and review text into a single combined text. The model will encode this combined text and it will output a single vector embedding.

To run this notebook, you will need to install: pandas, openai, transformers, plotly, matplotlib, scikit-learn, torch (transformer dep), torchvision, and scipy.

In [ ]:
!pip install tiktoken

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 KB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 KB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 KB 15.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.25.1
    Uninstalling requests-2.25.1:
      Successfully uninstalled requests-2.25.1


In [ ]:
pip install pandas

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 32.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 KB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 82.8 MB/s eta 0:00:00


In [ ]:
pip install plotly

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install matplotlib

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install scikit-learn

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install torch

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install torchvision

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install scipy

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
 pip install --upgrade pip

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 22.0.4
    Uninstalling pip-22.0.4:
      Successfully uninstalled pip-22.0.4


In [ ]:
!pip install openai

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 8.9 MB/s eta 0:00:00
  Created wheel for openai: filename=openai-0.27.1-py3-none-any.whl size=70091 sha256=9e1af47c153109dc0d8902bb9f75d35e523bdb291f97ea4191981f6b3d189c3b
  Stored in directory: /root/.cache/pip/wheels/1f/d1/75/8015df8f7ec8ba5422d8a45786cbb64d421872f488c09303fe
Successfully built openai


In [ ]:
# imports
import pandas as pd
import tiktoken

from openai.embeddings_utils import get_embedding

In [ ]:
# embedding model parameters
embedding_model = "text-embedding-ada-002"
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191


In [ ]:
input_datapath="/content/10pearls_sections_text_final.xlsx"
df = pd.read_excel(input_datapath)#, index_col=0)
df.head()
# df = df[["Time", "ProductId", "UserId", "Score", "Summary", "Text"]]
df=df[["title", "heading", "content"]]
df = df.dropna()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 144 entries, 0 to 144
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    144 non-null    object
 1   heading  144 non-null    object
 2   content  144 non-null    object
dtypes: object(3)
memory usage: 4.5+ KB


In [ ]:
# load & inspect dataset
# input_datapath = "./fine_food_reviews_1k.csv"  # to save space, we provide a pre-filtered dataset
# df = pd.read_csv(input_datapath, index_col=0)
# df = df[["Time", "ProductId", "UserId", "Score", "Summary", "Text"]]

df["combined"] = (
    "Title: " + df.title.str.strip() +"; Heading: " + df.heading.str.strip()+ "; Content: " + df.content.str.strip()
)
df.head(2)

,title,heading,content,combined,n_tokens
0,10Pearls Policies and Procedures,OUR MISSION,TenPearls (Private) Limited is founded on the ...,Title: 10Pearls Policies and Procedures; Headi...,106
1,Definitions in 10Pearls Policies and Procedures,DEFINITIONS,In this Manual the following words and express...,Title: Definitions in 10Pearls Policies and Pr...,39


In [ ]:
max_tokens

8000

In [ ]:
df["n_tokens"].head(10)

0    106
1     39
2     28
3     34
4     51
5     48
6     48
7     28
8     43
9     43
Name: n_tokens, dtype: int64

In [ ]:
# subsample to 1k most recent reviews and remove samples that are too long
top_n = 1000
# df = df.sort_values("Time").tail(top_n * 2)  # first cut to first 2k entries, assuming less than half will be filtered out
# df.drop("Time", axis=1, inplace=True)

encoding = tiktoken.get_encoding(embedding_encoding)

# omit reviews that are too long to embed
df["n_tokens"] = df.combined.apply(lambda x: len(encoding.encode(x)))
# df = df[df.n_tokens <= max_tokens].tail(top_n)
len(df)

144

In [ ]:
df[df.n_tokens <= max_tokens]

,title,heading,content,combined,n_tokens
0,10Pearls Policies and Procedures,OUR MISSION,TenPearls (Private) Limited is founded on the ...,Title: 10Pearls Policies and ProceduresOUR MIS...,106
1,Definitions in 10Pearls Policies and Procedures,DEFINITIONS,In this Manual the following words and express...,Title: Definitions in 10Pearls Policies and Pr...,39
2,Definitions in 10Pearls Policies and Procedures,Company,“Company” shall mean TenPearls (Private) Limited;,Title: Definitions in 10Pearls Policies and Pr...,28
3,Definitions in 10Pearls Policies and Procedures,PULSE,“PULSE” shall mean the Company online portal a...,Title: Definitions in 10Pearls Policies and Pr...,34
4,Definitions in 10Pearls Policies and Procedures,Employee,“Employee(s)” shall mean all employees of the ...,Title: Definitions in 10Pearls Policies and Pr...,51
...,...,...,...,...,...
140,APPENDIX J in 10Pearls Policies and Procedures,CEILING AMOUNT,Level\nManagers and Asst. Managers/Architects\...,Title: APPENDIX J in 10Pearls Policies and Pro...,85
141,APPENDIX K in 10Pearls Policies and Procedures,PURPOSE,The Company has designed LTIP to reward its el...,Title: APPENDIX K in 10Pearls Policies and Pro...,118
142,APPENDIX K in 10Pearls Policies and Procedures,PARTICIPANTS FOR THE LTIP UNITS,• The officers and other senior executive Conf...,Title: APPENDIX K in 10Pearls Policies and Pro...,526
143,APPENDIX K in 10Pearls Policies and Procedures,MINIMUM HOLDING PERIOD AND ACTUAL HOLDING PERIOD,The Minimum Holding Period for LTIP Units issu...,Title: APPENDIX K in 10Pearls Policies and Pro...,330


## 2. Get embeddings and save them for future reuse

In [ ]:
import openai
openai.api_key=''

In [ ]:
df.loc[0:40]

,title,heading,content,combined,n_tokens
0,10Pearls Policies and Procedures,OUR MISSION,TenPearls (Private) Limited is founded on the ...,Title: 10Pearls Policies and Procedures; Headi...,106
1,Definitions in 10Pearls Policies and Procedures,DEFINITIONS,In this Manual the following words and express...,Title: Definitions in 10Pearls Policies and Pr...,39
2,Definitions in 10Pearls Policies and Procedures,Company,“Company” shall mean TenPearls (Private) Limited;,Title: Definitions in 10Pearls Policies and Pr...,28
3,Definitions in 10Pearls Policies and Procedures,PULSE,“PULSE” shall mean the Company online portal a...,Title: Definitions in 10Pearls Policies and Pr...,34
4,Definitions in 10Pearls Policies and Procedures,Employee,“Employee(s)” shall mean all employees of the ...,Title: Definitions in 10Pearls Policies and Pr...,51
5,Definitions in 10Pearls Policies and Procedures,Full-Time Employee,“Full-Time Employee(s)” shall mean Employee(s)...,Title: Definitions in 10Pearls Policies and Pr...,48
6,Definitions in 10Pearls Policies and Procedures,Fixed Term Contractual Employee,“Fixed Term Contractual Employee(s)” shall mea...,Title: Definitions in 10Pearls Policies and Pr...,48
7,Definitions in 10Pearls Policies and Procedures,HR,“HR” shall mean the Human Resource Department ...,Title: Definitions in 10Pearls Policies and Pr...,28
8,Definitions in 10Pearls Policies and Procedures,Half Day,“Half Day” shall mean at least four and a half...,Title: Definitions in 10Pearls Policies and Pr...,43
9,Definitions in 10Pearls Policies and Procedures,Interns,“Interns” shall mean person(s) engaged for wor...,Title: Definitions in 10Pearls Policies and Pr...,43


In [ ]:
df.loc[0:30, 'embeddings']=0

In [ ]:
df.loc[0:30, 'embeddings']=df[0:30].combined.apply(lambda x: get_embedding(x, engine=embedding_model))

In [ ]:
df.loc[30:60, 'embeddings']=df[30:60].combined.apply(lambda x: get_embedding(x, engine=embedding_model))

In [ ]:
df.loc[60:90, 'embeddings']=df[60:90].combined.apply(lambda x: get_embedding(x, engine=embedding_model))

In [ ]:
df.loc[90:120, 'embeddings']=df[90:120].combined.apply(lambda x: get_embedding(x, engine=embedding_model))

In [ ]:
df.loc[120:, 'embeddings']=df[120:].combined.apply(lambda x: get_embedding(x, engine=embedding_model))

In [ ]:
output_datapath="/content/10pearls_sections_text_embeddings.xlsx"

In [ ]:
df.to_excel(output_datapath)

In [ ]:
for i in range(0, 145, 20):
  df.loc[i:i+20, 'embeddings']=df[i:i+20].combined.apply(lambda x: get_embedding(x, engine=embedding_model))
  #df.to_csv("data/fine_food_reviews_with_embeddings_1k.csv")


RetryError: ignored